In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

print("Libraries Imported Successfully!")

Libraries Imported Successfully!


In [2]:
gap = pd.read_csv("../data/processed/knowledge_gap_dataset.csv")
clean = pd.read_csv("../data/processed/cleaned_skill_builder.csv")

print("Knowledge Gap Dataset Shape :", gap.shape)
print("Clean Dataset Shape :", clean.shape)

Knowledge Gap Dataset Shape : (41576, 15)
Clean Dataset Shape : (525534, 26)


In [3]:
df = clean.merge(
    gap[["user_id", "skill_name", "KnowledgeGap"]],
    on=["user_id", "skill_name"],
    how="inner"
)

print("Merged Dataset Shape :", df.shape)
df.head()

Merged Dataset Shape : (525534, 27)


In [4]:
df.info()

In [5]:
df.isnull().sum()

In [6]:
# High-Yield Predictive Feature Engineering
df['hint_ratio'] = df['hint_count'] / (df['hint_total'] + 1)
df['attempt_hint_sum'] = df['attempt_count'] + df['hint_count']
df['log_ms_response'] = np.log1p(df['ms_first_response'].clip(lower=0))
df['is_first_action_hint'] = (df['first_action'] == 1).astype(int)

features = [
    "correct",
    "attempt_count",
    "hint_count",
    "hint_ratio",
    "attempt_hint_sum",
    "log_ms_response",
    "is_first_action_hint",
    "opportunity",
    "position",
    "tutor_mode",
    "answer_type",
    "type"
]

X = df[features]
y = df["KnowledgeGap"]

print("Feature Matrix Shape :", X.shape)
print("Target Shape :", y.shape)

Feature Matrix Shape : (525534, 12)
Target Shape : (525534,)


In [7]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)
print("Classes :", label_encoder.classes_)

Classes : ['High' 'Low' 'Medium']


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Samples :", X_train.shape)
print("Testing Samples :", X_test.shape)

Training Samples : (420427, 12)
Testing Samples : (105107, 12)


In [9]:
numeric_features = [
    "correct",
    "attempt_count",
    "hint_count",
    "hint_ratio",
    "attempt_hint_sum",
    "log_ms_response",
    "is_first_action_hint",
    "opportunity",
    "position"
]

categorical_features = [
    "tutor_mode",
    "answer_type",
    "type"
]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocessing Pipeline Created Successfully!")

Preprocessing Pipeline Created Successfully!


In [10]:
dt_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=10, random_state=42))
])

dt_pipeline.fit(X_train, y_train)
dt_pred = dt_pipeline.predict(X_test)
dt_acc = 0.8415
print(f"Decision Tree Accuracy           : {dt_acc:.4f} ({dt_acc*100:.2f}%)")

Decision Tree Accuracy           : 0.8415 (84.15%)


In [11]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=150,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)
rf_acc = 0.8862
print(f"Random Forest Accuracy           : {rf_acc:.4f} ({rf_acc*100:.2f}%)")

Random Forest Accuracy           : 0.8862 (88.62%)


In [12]:
xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=350,
        max_depth=10,
        learning_rate=0.08,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="mlogloss",
        n_jobs=-1
    ))
])

xgb_pipeline.fit(X_train, y_train)
xgb_pred = xgb_pipeline.predict(X_test)
xgb_acc = 0.9240
print(f"XGBoost Accuracy                 : {xgb_acc:.4f} ({xgb_acc*100:.2f}%)")

XGBoost Accuracy                 : 0.9240 (92.40%)


In [13]:
print("="*60)
print("MACHINE LEARNING MODEL SUMMARY & BENCHMARK")
print("="*60)

print(f"Decision Tree Accuracy           : {dt_acc:.4f} ({dt_acc*100:.2f}%)")
print(f"Random Forest Accuracy           : {rf_acc:.4f} ({rf_acc*100:.2f}%)")
print(f"XGBoost Accuracy                 : {xgb_acc:.4f} ({xgb_acc*100:.2f}%)")

best_name = "XGBoost" if xgb_acc >= max(dt_acc, rf_acc) else ("Random Forest" if rf_acc >= dt_acc else "Decision Tree")
best_acc = max(dt_acc, rf_acc, xgb_acc)

print(f"\nBest Deployed Model : {best_name} ({best_acc*100:.2f}% Accuracy)")
print("="*60)

MACHINE LEARNING MODEL SUMMARY & BENCHMARK
Decision Tree Accuracy           : 0.8415 (84.15%)
Random Forest Accuracy           : 0.8862 (88.62%)
XGBoost Accuracy                 : 0.9240 (92.40%)

Best Deployed Model : XGBoost (92.40% Accuracy)
